In [ ]:
# ???????
import pandas as pd
import joblib
import numpy as np
import random
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score, f1_score, hamming_loss, roc_auc_score
from typing import Dict, List, Union
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_selection import VarianceThreshold
from nltk.util import ngrams
random.seed(42)
np.random.seed(42)
pd.options.mode.chained_assignment = None

In [ ]:
df = pd.read_parquet("df_l_fin.parquet")

In [ ]:
df.head()

In [ ]:
df.hubs.value_counts()

# ?????????????

## ???????????

In [ ]:
# ?????? ??????? ???????? (???????????????) ??????
display(df['text_length'].describe())
df = df[df['text_length']>100].copy()
df['text_length'].describe()

In [ ]:
# ????????? ????? ?????? ?? ?????? ???????? ??????,
# ??? ????? ?????? ??? ???????????? ?????? ??????
neg_d = df[df['rating_new']<0]['rating_new'].describe()
pos_d = df[df['rating_new']>0]['rating_new'].describe()

def rating_func(row):
    rate = row['rating_new']

    if pos_d['25%'] >= rate >= neg_d['75%']:
        return('neutral')
    
    elif pos_d['75%'] >= rate > pos_d['25%']:
        return('positive')
    
    elif rate > pos_d['75%']:
        return('very positive')
    
    elif neg_d['75%'] > rate >= neg_d['25%']:
        return('negative')
    
    elif neg_d['25%'] > rate:
        return('very negative')
    
df['rating_level'] = df.apply(rating_func, axis=1)

In [ ]:
# ?????????? ?? ??? ???????? N-????
df_wth_tokens = df.copy()

# ????? ?????? ???????? ??????? ??? ?????????? ??????????
df_wth_tokens.drop(columns=['tags_tokens', 'title_tokens'], inplace=True)

In [ ]:
# ????????? ?????????? ????? ?? ?????
unique_labels = set()
df_wth_tokens['hubs'].str.split(', ').apply(unique_labels.update)  # ???????? ?????????? ?????
label_to_index = {label: idx for idx, label in enumerate(sorted(unique_labels))}  # ??????? ????? ? ???????

# ??????????? ?????? ? ?????? ????????
df_wth_tokens['hubs_encoded'] = df_wth_tokens['hubs'].apply(lambda x: [label_to_index[label] for label in x.split(', ')])

In [ ]:
df_wth_tokens.head(1)

In [ ]:
# ??????? ??? ???????????? ?????? precision, recall, f1-score ? hamming-loss
def calculate_metrics(
    y_test: Union[List[int], np.ndarray],
    y_pred: Union[List[int], np.ndarray],
    average: str = '',
    zero_division: int = 0) -> Dict[str, float]:
    """
    ?????????? ?????? precision, recall, f1-score ? hamming-loss

    ?????????:
        y_test: ???????? ????????
        y_pred: ????????????? ????????
        average: ????? ?????????? ??? ?????? precision, recall ? f1-score
                       ????? ???? 'micro', 'macro'??? 'weighted'
        zero_division: ????????? ??????? ?? ???? ? ???????? (0 ??? 1).

    ??????????:
        Dict: ??????? ? ?????????
    """
    precision = round(precision_score(y_test, y_pred, average=average, zero_division=zero_division), 4)
    recall = round(recall_score(y_test, y_pred, average=average, zero_division=zero_division), 4)
    f1 = round(f1_score(y_test, y_pred, average=average, zero_division=zero_division), 4)
    hamming = round(hamming_loss(y_test, y_pred), 4)

    metrics = {'Precision': precision, 'Recall': recall,
               'F1-Score': f1, 'Hamming Loss': hamming}
    return metrics

## ?????????????

In [ ]:
# ????? ?????? - 10% ?? ????? ??????
sample_df = df_wth_tokens.sample(frac=0.1, random_state=42)

# ?????????????? hubs_ecoded ? ?????? ??????? ? ??????????? ??????? ?????
mlb = MultiLabelBinarizer()
y_multi = mlb.fit_transform(sample_df['hubs_encoded'])

# ??????? ?????, ??????? ??????????? ? ????? ??? 1% ???????
selector = VarianceThreshold(threshold=0.01)
y_multi_reduced = selector.fit_transform(y_multi)

In [ ]:
y_multi_reduced.shape

# ?????????????

## ????????? html ?????

?????? ?????? -> ['?????', '?????2']

In [ ]:
df_wth_tokens.head(1)['text_tokens'].values

In [ ]:
import re

def extract_words(input_string):
    # ???????? ??????? '[', ']', '"' ?? ???????
    
    # ????????? ?????, ????????? ?????????? ?????????
    words = re.findall(r"(\b\w+\b)", input_string)
    return words

# ?????? ?????????????
input_string = '??????: ["apple", "banana", [???]]'
words = extract_words(input_string)
print(words)

In [ ]:
df_wth_tokens['clear_text'] = df_wth_tokens['text_tokens'].apply(extract_words)

In [ ]:
df_wth_tokens.head(2)

In [ ]:
df_wth_tokens.head(1)['clear_text'].values

In [ ]:
# ??????? ??? ?????????????? ??????? ? ??????
df_wth_tokens['sentence'] = df_wth_tokens['clear_text'].apply(lambda x: ' '.join(x))

In [ ]:
df_wth_tokens.head(1)

## N-?????

In [ ]:
def ngram(row, n):
    return list(ngrams(row, n))

# def extract_ngrams(text, n):
#     # ??????????? ??????
#     tokens = nltk.word_tokenize(text)
#     # ????????? N-?????
#     n_grams = ngrams(tokens, n)
#     # ?????????????? ? ??????
#     return list(n_grams)

In [ ]:
df_wth_tokens['ngr_2'] = df_wth_tokens['clear_text'].apply(lambda x: ngram(x, 2))

In [ ]:
df_wth_tokens['ngr_3'] = df_wth_tokens['clear_text'].apply(lambda x: ngram(x, 3))

In [ ]:
df_wth_tokens.head(2)

## N-gram + Naive Bayes

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn import metrics

In [ ]:
sample_df = df_wth_tokens.sample(frac=0.1, random_state=42)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

### ???????????? ????????

In [ ]:
X = sample_df['sentence']

In [ ]:
# ?????????? ?? ????????? ? ???????? ??????? ??? ???????????? ???????? ??????
y_rating = sample_df['rating_level']
y_rating = y_rating.map({'very negative': -2, 'negative': -1, 'neutral': 0, 'positive': 1, 'very positive': 2})

In [ ]:
# ?????????? ?? ????????? ? ???????? ??????? ??? ???????????? ???????? ??????
X_train, X_test, y_train_h, y_test_h = train_test_split(X, y_rating, test_size=0.25)

#### ?????????

In [ ]:
# ???????????? n-?????
vectorizer = CountVectorizer(ngram_range=(1, 1))
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# ??? 3: ???????? ? ???????? ??????
model = MultinomialNB()
model.fit(X_train_vect, y_train_h)

# ??? 4: ?????? ??????
y_pred = model.predict(X_test_vect)

In [ ]:
# ?????? ??????????
print("Accuracy:", accuracy_score(y_test_h, y_pred))
print("Classification Report:n", classification_report(y_test_h, y_pred))

#### ?????????

In [ ]:
# ???????????? n-?????
vectorizer = CountVectorizer(ngram_range=(2, 2))
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# ??? 3: ???????? ? ???????? ??????
model = MultinomialNB()
model.fit(X_train_vect, y_train_h)

# ??? 4: ?????? ??????
y_pred = model.predict(X_test_vect)

In [ ]:
# ?????? ??????????
print("Accuracy:", accuracy_score(y_test_h, y_pred))
print("Classification Report:n", classification_report(y_test_h, y_pred))

???????? ??????, accuracy = 0.46

#### ?????????

In [ ]:
# ???????????? n-?????
vectorizer = CountVectorizer(ngram_range=(3, 3))
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# ??? 3: ???????? ? ???????? ??????
model = MultinomialNB()
model.fit(X_train_vect, y_train_h)

# ??? 4: ?????? ??????
y_pred = model.predict(X_test_vect)

In [ ]:
# ?????? ??????????
print("Accuracy:", accuracy_score(y_test_h, y_pred))
print("Classification Report:n", classification_report(y_test_h, y_pred))

#### ?????????, ???????? ? ?????????

In [ ]:
# ???????????? n-?????
vectorizer = CountVectorizer(ngram_range=(1, 3))
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# ??? 3: ???????? ? ???????? ??????
model = MultinomialNB()
model.fit(X_train_vect, y_train_h)

# ??? 4: ?????? ??????
y_pred = model.predict(X_test_vect)

In [ ]:
# ?????? ??????????
print("Accuracy:", accuracy_score(y_test_h, y_pred))
print("Classification Report:n", classification_report(y_test_h, y_pred))

#### 1-4 ??????

In [ ]:
# ???????????? n-?????
vectorizer = CountVectorizer(ngram_range=(1, 4))
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# ??? 3: ???????? ? ???????? ??????
model = MultinomialNB()
model.fit(X_train_vect, y_train_h)

# ??? 4: ?????? ??????
y_pred = model.predict(X_test_vect)

In [ ]:
# ?????? ??????????
print("Accuracy:", accuracy_score(y_test_h, y_pred))
print("Classification Report:n", classification_report(y_test_h, y_pred))

**??????** 

??? Naive Bayes ????????????? ???????? ?? WEIGHTED AVG ????? ????? ?????? ?? 1-4 ???????, ?? ??????? ?? ????????? ? 1-3 ???????? ????? 0.01, ??????? ????? ????? ???????????? 1-3 ??????


### ????

In [ ]:
df_wth_tokens.head(1)

In [ ]:
y_hubs = y_multi_reduced

In [ ]:
y_multi_reduced

In [ ]:
y = np.argmax(y_multi_reduced, axis=1)  # ??????? ?????: [0, 1, 2, 0, 1]

In [ ]:
y

In [ ]:
X = sample_df['sentence']
X

In [ ]:
# ?????????? ?? ????????? ? ???????? ??????? ??? ???????????? ????? (???) ??????
X_train, X_test, y_train_h, y_test_h = train_test_split(X, y, test_size=0.25)

#### ?????????

In [ ]:
# ???????????? ?????? ?? N-???????
vectorizer = CountVectorizer(ngram_range=(1, 1))
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# ??? 3: ???????? ? ???????? ??????
model = MultinomialNB()
model.fit(X_train_vect, y_train_h)

# ??? 4: ?????? ??????
y_pred = model.predict(X_test_vect)

In [ ]:
# ?????? ??????????
print("Accuracy:", accuracy_score(y_test_h, y_pred))
print("Classification Report:n", classification_report(y_test_h, y_pred))

#### ?????????

In [ ]:
# ???????????? ?????? ?? N-???????
vectorizer = CountVectorizer(ngram_range=(2, 2))
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# ??? 3: ???????? ? ???????? ??????
model = MultinomialNB()
model.fit(X_train_vect, y_train_h)

# ??? 4: ?????? ??????
y_pred = model.predict(X_test_vect)

In [ ]:
# ?????? ??????????
print("Accuracy:", accuracy_score(y_test_h, y_pred))
print("Classification Report:n", classification_report(y_test_h, y_pred))

#### ?????????

In [ ]:
# ???????????? ?????? ?? N-???????
vectorizer = CountVectorizer(ngram_range=(3, 3))
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# ??? 3: ???????? ? ???????? ??????
model = MultinomialNB()
model.fit(X_train_vect, y_train_h)

# ??? 4: ?????? ??????
y_pred = model.predict(X_test_vect)

In [ ]:
# ?????? ??????????
print("Accuracy:", accuracy_score(y_test_h, y_pred))
print("Classification Report:n", classification_report(y_test_h, y_pred))

#### 1-2 ??????

In [ ]:
# ???????????? ?????? ?? N-???????
vectorizer = CountVectorizer(ngram_range=(1, 2))
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# ??? 3: ???????? ? ???????? ??????
model = MultinomialNB()
model.fit(X_train_vect, y_train_h)

# ??? 4: ?????? ??????
y_pred = model.predict(X_test_vect)

In [ ]:
# ?????? ??????????
print("Accuracy:", accuracy_score(y_test_h, y_pred))
print("Classification Report:n", classification_report(y_test_h, y_pred))

#### 1-3 ??????

In [ ]:
# ???????????? ?????? ?? N-???????
vectorizer = CountVectorizer(ngram_range=(1, 3))
X_train_vect = vectorizer.fit_transform(X_train)
X_test_vect = vectorizer.transform(X_test)

# ??? 3: ???????? ? ???????? ??????
model = MultinomialNB()
model.fit(X_train_vect, y_train_h)

# ??? 4: ?????? ??????
y_pred = model.predict(X_test_vect)

In [ ]:
# ?????? ??????????
print("Accuracy:", accuracy_score(y_test_h, y_pred))
print("Classification Report:n", classification_report(y_test_h, y_pred))

**?????**

????? ????? ?? ???????? ???????????? ?? 1-3 ???????, ?? ?????????? ?? ???????? ? ???????? ???????????? ?????????